In [ ]:
library(DESeq2)
library(tidyverse)
library(airway) 
library(repr)

In [ ]:
df <- read.table("rawCounts.tsv",sep='\t',header=T,row.names="Geneid")

In [7]:
colnames(df) <- c('Chr', 'Start', 'End', 'Strand', 'Length','EPN_1','EPN_2','Huh7_Bnase_1','Huh7_Bnase_2','Huh7_Mock_1','Huh7_Mock_2','IGF2_1','IGF2_2','IGF2_D1','IGF2_D2','IGF2_Dihi_1','IGF2_Dihi_2','IGF2_DR1','IGF2_DR2','IGF2_L03_1','IGF2_L03_2','IGF2_R1','IGF2_R2','PAP_1','PAP_2','YTH_1','YTH_2') 

In [8]:
asint <- function(x) {  
  as.integer(round(x,digits=0))}

In [9]:
dim(df)

[1] 61636    27

In [10]:
df[c('EPN_1','EPN_2','IGF2_1','IGF2_2','PAP_1','PAP_2','YTH_1','YTH_2')]

,EPN_1,EPN_2,IGF2_1,IGF2_2,PAP_1,PAP_2,YTH_1,YTH_2
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ENSG00000223972.5,4.5,0,0.0,1.0,8.0,2.5,2.0,1.5
ENSG00000227232.5,17.5,21,8.0,11.0,171.5,142.5,80.0,48.0
ENSG00000278267.1,13.0,2,0.0,0.0,3.5,6.0,0.0,0.0
ENSG00000243485.5,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
ENSG00000284332.1,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
ENSG00000237613.2,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
ENSG00000268020.3,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
ENSG00000240361.2,6.0,10,0.0,2.0,0.0,2.0,0.0,0.0
ENSG00000186092.7,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
df_raw<-as.data.frame(lapply(df[c('EPN_1','EPN_2','IGF2_1','IGF2_2','PAP_1','PAP_2','YTH_1','YTH_2')],asint),row.names=row.names(df))

In [12]:
df_raw<-df_raw[rowSums(df_raw)>1,]

In [13]:
colData=data.frame(id = colnames(df_raw),
    group = c('EPN','EPN','IGF2BP1','IGF2BP1','PABPC1','PABPC1','YTHDF2','YTHDF2'))
colData$group <- factor(colData$group, levels = c('EPN','IGF2BP1','PABPC1','YTHDF2'))

In [14]:
dds <- DESeqDataSetFromMatrix(countData = df_raw, colData = colData, design = ~ group)

In [18]:
df_raw[c(56418:56485), ]

,EPN_1,EPN_2,IGF2_1,IGF2_2,PAP_1,PAP_2,YTH_1,YTH_2
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
ERCC-00002-DQ459430,9900,7964,996,1390,7448,6326,602,618
ERCC-00003-DQ516784,898,788,76,98,554,444,40,28
ERCC-00004-DQ516752,1940,1636,128,166,1036,914,66,66
ERCC-00009-DQ668364,238,188,12,22,198,150,14,10
ERCC-00013-EF011062,2,0,0,0,0,0,0,0
ERCC-00014-DQ875385,2,6,2,0,0,4,0,0
ERCC-00019-DQ883651,4,14,0,2,2,6,0,0
ERCC-00022-DQ855004,86,116,8,22,76,70,6,2
ERCC-00025-DQ883689,102,76,10,22,76,66,4,4


In [19]:
dds <- estimateSizeFactors(dds,controlGenes=56418:56485)

In [20]:
dds_norm <- DESeq(dds)

using pre-existing size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [21]:
as.matrix(dds_norm$sizeFactor)

EPN_1,4.1336164
EPN_2,3.3893268
IGF2_1,0.3611171
IGF2_2,0.5162871
PAP_1,2.8708780
PAP_2,2.5103846
YTH_1,0.2488209
YTH_2,0.2091668


In [22]:
normalized_counts <- counts(dds_norm,normalized=T)
write.csv(normalized_counts,file="result/normalized_counts_spikein.csv")

In [23]:
res_1  = results(dds_norm, contrast=c("group",'IGF2BP1','EPN'))
res_2  = results(dds_norm, contrast=c("group",'PABPC1','EPN'))
res_3  = results(dds_norm, contrast=c("group",'YTHDF2','EPN'))

In [ ]:
id2symbol=read.table("ref/gene_id_type.tsv",sep='\t')
colnames(id2symbol)<-c("gene_id",'gene_symbol','gene_type')
row.names(id2symbol)<-id2symbol$gene_id
id2symbol<-subset(id2symbol,select=-gene_id)

In [ ]:
id2symbo2=read.table("ref/ERCC.bed",sep='\t',row.names = 1)
colnames(id2symbo2)<-c('gene_symbol',"gene_type")

In [26]:
id2symbol<-rbind(id2symbol,id2symbo2)

In [27]:
normalized_counts1 <-merge(id2symbol,data.frame(normalized_counts ),all.y=T,by='row.names')

In [28]:
write.csv(normalized_counts1,file="result/normalized_counts_spikein_gene2.csv")

In [29]:
res_1 <-merge(data.frame(res_1 ),id2symbol,all.x=T,by='row.names')
res_2 <-merge(data.frame(res_2 ),id2symbol,all.x=T,by='row.names')
res_3 <-merge(data.frame(res_3 ),id2symbol,all.x=T,by='row.names')

In [30]:
res_1 = res_1 [order(res_1 $pvalue),]
res_2 = res_2 [order(res_2 $pvalue),]
res_3 = res_3 [order(res_3 $pvalue),]

In [31]:
write.csv(res_1 , file="result/IGF2BP1_vs_EPN_DESeq2.csv",row.names=T)
write.csv(res_2 , file="result/PABPC1_vs_EPN_DESeq2.csv",row.names=T)
write.csv(res_3 , file="result/YTHDF2_vs_EPN_DESeq2.csv",row.names=T)

---------

In [22]:
library(clusterProfiler)
library(DOSE)
library(org.Hs.eg.db)
library(enrichplot)
library(stringr)



clusterProfiler v4.7.1.003  For help: https://yulab-smu.top/biomedical-knowledge-mining-book/

If you use clusterProfiler in published research, please cite:
T Wu, E Hu, S Xu, M Chen, P Guo, Z Dai, T Feng, L Zhou, W Tang, L Zhan, X Fu, S Liu, X Bo, and G Yu. clusterProfiler 4.0: A universal enrichment tool for interpreting omics data. The Innovation. 2021, 2(3):100141


Attaching package: 'clusterProfiler'


The following object is masked from 'package:purrr':

    simplify


The following object is masked from 'package:IRanges':

    slice


The following object is masked from 'package:S4Vectors':

    rename


The following object is masked from 'package:stats':

    filter


DOSE v3.25.0.002  For help: https://yulab-smu.top/biomedical-knowledge-mining-book/

If you use DOSE in published research, please cite:
Guangchuang Yu, Li-Gen Wang, Guang-Rong Yan, Qing-Yu He. DOSE: an R/Bioconductor package for Disease Ontology Semantic and Enrichment analysis. Bioinformatics 2015, 31(4):608

In [23]:
function_anlysis <- function(DEseq2,sample_name) {
    DEseq2$ENSEMBL<-str_split(DEseq2$Row.names,"\\.",n=2,simplify = TRUE)[,1]
    gene.df <- bitr(DEseq2$ENSEMBL,
               fromType = "ENSEMBL",
               toType = c("SYMBOL","ENTREZID"),
               OrgDb = org.Hs.eg.db)
    DEseq2_merge=merge(DEseq2,gene.df,by.y="ENSEMBL",by.x="ENSEMBL")
    genelist <- DEseq2_merge$log2FoldChange
    names(genelist)<-DEseq2_merge$ENTREZID
    genelist <- sort(genelist, decreasing = TRUE)
    gsea <- gseGO(
        geneList = genelist, 
        OrgDb   = org.Hs.eg.db,
        ont = "BP", eps=0, 
        minGSSize = 15,
        maxGSSize = 1000,
        pvalueCutoff = 0.05,
        pAdjustMethod  = "BH")
    write.csv(gsea,file=paste("result/GSEA/",sample_name,"_GO_GSEA.csv", sep = ""), quote=T, sep=",", row.names=F)
    kk <- gseKEGG(
        geneList  = genelist,
        keyType  = 'kegg',
        organism = 'hsa',
        eps=0,
        minGSSize = 15,
        maxGSSize = 1000,
        pvalueCutoff = 0.05,
        pAdjustMethod  = "BH"
    )
    write.csv(kk,file=paste("result/KEGG/",sample_name,"_KEGG_GSEA.csv", sep = ""), quote=T, sep=",", row.names=F)
    ego_up <- enrichGO(
        gene = (DEseq2 %>% filter(log2FoldChange>  1, padj<0.05 ))$ENSEMBL,
        OrgDb = org.Hs.eg.db, 
        keyType = 'ENSEMBL', 
        ont = "BP", 
        minGSSize = 15,
        pvalueCutoff = 0.05,
        readable = T)
    ego_down <- enrichGO(
        gene = (DEseq2 %>% filter(log2FoldChange< -1, padj<0.05 ))$ENSEMBL,
        OrgDb = org.Hs.eg.db, 
        keyType = 'ENSEMBL', 
        ont = "BP", 
        minGSSize = 15,
        pvalueCutoff = 0.05,
        readable = T)
    write.csv(simplify(ego_up,cutoff=0.7, by="p.adjust", select_fun=min),  file=paste("result/GO/",sample_name,"_upGO.csv", sep = ""), quote=T, sep=",", row.names=F)
    write.csv(simplify(ego_down,cutoff=0.7, by="p.adjust", select_fun=min),file=paste("result/GO/",sample_name,"_downGO.csv", sep = ""), quote=T, sep=",", row.names=F)
    return(list(gsea=gsea,kk=kk,ego_up=ego_up,ego_down=ego_down))
}

In [ ]:
fun_res_1 <- function_anlysis(res_1,"M36_vs_WT")
fun_res_2 <- function_anlysis(res_2,"M37_vs_WT")
fun_res_3 <- function_anlysis(res_3,"M40_vs_WT")

In [ ]:
options(repr.plot.width=9,repr.plot.height=6)
ps<-gseaplot2(fun_res_2$kk, geneSetID=c("hsa04668","hsa04657","hsa04064","hsa04062"),title = "0-3hr - TNFa vs DMSO",base_size = 16,pvalue_table = T,rel_heights = c(1.5, 0.8, 0.75))
ps